# **Stratified Sampling untuk Anotasi (incremental, anti-duplikat)**

Mengambil sampel **stratified per aspek** dari `sentences_with_aspect.csv` untuk dianotasi sentimennya.

Desain **incremental**: bisa dijalankan berkali-kali untuk menambah batch tanpa memilih ulang kalimat yang sudah pernah terpilih. Kuncinya:
- setiap kalimat punya `id_kalimat` unik;
- daftar id yang sudah terpilih disimpan di `data/annotation/sudah_terpilih.csv`;
- batch baru selalu **mengecualikan** id yang sudah ada -> tidak ada duplikat;
- `random_state` tetap -> reproducible.

**Lainnya** (outlier/topik tak jelas) dikeluarkan dari sampling.

In [ ]:
from pathlib import Path
import pandas as pd

SEED = 42
ANNDIR = Path("data/annotation")
ANNDIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv("data/processed/sentences_with_aspect.csv", dtype={"id_artikel": str})
# kandidat anotasi: semua kalimat KECUALI 'Lainnya'
kandidat = df[df["nama_aspek"] != "Lainnya"].copy()
assert kandidat["id_kalimat"].is_unique, "id_kalimat harus unik"
print(f"Total kalimat: {len(df):,} | kandidat (tanpa Lainnya): {len(kandidat):,}")
print(f"Jumlah aspek: {kandidat['nama_aspek'].nunique()}")

## Muat id yang SUDAH pernah terpilih (checkpoint)

Pertama kali dijalankan, file ini belum ada -> set kosong. Pada run berikutnya, berisi semua id batch sebelumnya sehingga tidak terpilih lagi.

In [ ]:
CHECKPOINT = ANNDIR / "sudah_terpilih.csv"
if CHECKPOINT.exists():
    sudah = pd.read_csv(CHECKPOINT, dtype={"id_kalimat": str})
    sudah_ids = set(sudah["id_kalimat"])
else:
    sudah_ids = set()
print(f"Sudah terpilih sebelumnya: {len(sudah_ids):,} kalimat")

## Parameter batch

`TARGET_TOTAL` = total pasangan yang diinginkan **kumulatif** (mis. 3000). Kalau sudah ada N terpilih, batch ini menambah hingga mencapai target. Alokasi **stratified proporsional** per aspek, dengan **minimum per aspek** agar aspek kecil tetap kebagian.

In [ ]:
TARGET_TOTAL = 1500   # target kumulatif; naikkan (mis. 3500, 4000) untuk menambah batch berikutnya
MIN_PER_ASPEK = 80    # minimum sampel tiap aspek (dibatasi ketersediaan)

perlu_tambah = max(0, TARGET_TOTAL - len(sudah_ids))
print(f"Target kumulatif: {TARGET_TOTAL} | sudah ada: {len(sudah_ids)} | perlu tambah: {perlu_tambah}")

In [ ]:
# sisa kandidat = kandidat yang BELUM pernah terpilih (kunci anti-duplikat)
sisa = kandidat[~kandidat["id_kalimat"].isin(sudah_ids)].copy()
aspek_list = sorted(sisa["nama_aspek"].unique())
n_aspek = len(aspek_list)

# alokasi proporsional terhadap ukuran aspek di sisa, dengan lantai MIN_PER_ASPEK
ukuran = sisa["nama_aspek"].value_counts()
alok = {}
if perlu_tambah > 0:
    dasar = {a: max(MIN_PER_ASPEK, round(perlu_tambah * ukuran[a] / len(sisa))) for a in aspek_list}
    # batasi ke ketersediaan tiap aspek
    dasar = {a: min(dasar[a], int(ukuran[a])) for a in aspek_list}
    # sesuaikan agar total ~ perlu_tambah (potong kelebihan dari aspek terbesar)
    total = sum(dasar.values())
    for a in sorted(aspek_list, key=lambda x: -ukuran[x]):
        if total <= perlu_tambah:
            break
        bisa_kurang = min(dasar[a] - MIN_PER_ASPEK, total - perlu_tambah)
        if bisa_kurang > 0:
            dasar[a] -= bisa_kurang; total -= bisa_kurang
    alok = dasar
print("Alokasi sampel batch ini per aspek:")
for a in aspek_list:
    print(f"  {a:34} {alok.get(a,0):>4} (tersedia {int(ukuran[a])})")
print(f"Total batch ini: {sum(alok.values())}")

## Ambil sampel batch (stratified, acak dengan seed)

In [ ]:
batch_parts = []
for a in aspek_list:
    n = alok.get(a, 0)
    if n > 0:
        pool = sisa[sisa["nama_aspek"] == a]
        batch_parts.append(pool.sample(n=min(n, len(pool)), random_state=SEED))
batch = pd.concat(batch_parts).sample(frac=1, random_state=SEED).reset_index(drop=True) if batch_parts else kandidat.iloc[0:0]

# nomor batch otomatis
existing_batches = sorted(ANNDIR.glob("to_annotate_batch*.csv"))
batch_no = len(existing_batches) + 1
print(f"Batch ke-{batch_no}: {len(batch)} kalimat")
assert batch["id_kalimat"].is_unique
assert not (set(batch["id_kalimat"]) & sudah_ids), "ADA DUPLIKAT dengan batch sebelumnya!"
print("Verifikasi anti-duplikat: OK (tidak ada irisan dengan yang sudah terpilih)")

## Bagi acak ke 2 anotator + sisihkan irisan untuk Cohen's Kappa

In [ ]:
N_OVERLAP = 150   # irisan yang dilabel KEDUA anotator (untuk Kappa)

overlap = batch.iloc[:N_OVERLAP].copy()
sisa_batch = batch.iloc[N_OVERLAP:].copy()
half = len(sisa_batch) // 2
porsi_A = sisa_batch.iloc[:half].copy()
porsi_B = sisa_batch.iloc[half:].copy()

def sheet(frame, anotator):
    out = frame[["id_kalimat", "id_artikel", "bank", "timestamp", "kalimat", "nama_aspek", "kategori_fiqa"]].copy()
    out["anotator"] = anotator
    out["sentimen"] = ""      # diisi manual: positif / netral / negatif
    #out["aspek_salah"] = ""   # diisi 'x' bila aspek jelas keliru
    #out["catatan"] = ""
    return out

anotator1 = pd.concat([sheet(porsi_A, "A"), sheet(overlap, "A")], ignore_index=True)
anotator2 = pd.concat([sheet(porsi_B, "B"), sheet(overlap, "B")], ignore_index=True)
print(f"Anotator A: {len(anotator1)} (termasuk {len(overlap)} irisan)")
print(f"Anotator B: {len(anotator2)} (termasuk {len(overlap)} irisan)")

## Simpan sheet anotasi + perbarui checkpoint

In [ ]:
anotator1.to_csv(ANNDIR / f"to_annotate_batch{batch_no}_A.csv", index=False, encoding="utf-8-sig")
anotator2.to_csv(ANNDIR / f"to_annotate_batch{batch_no}_B.csv", index=False, encoding="utf-8-sig")
overlap[["id_kalimat"]].to_csv(ANNDIR / f"overlap_batch{batch_no}.csv", index=False)

# perbarui checkpoint: tambahkan id batch ini (anti-duplikat utk run berikutnya)
ids_baru = pd.DataFrame({"id_kalimat": batch["id_kalimat"], "batch": batch_no})
if CHECKPOINT.exists():
    ids_baru = pd.concat([pd.read_csv(CHECKPOINT, dtype={"id_kalimat": str}), ids_baru], ignore_index=True)
ids_baru.drop_duplicates("id_kalimat").to_csv(CHECKPOINT, index=False)
print(f"Disimpan sheet batch {batch_no} untuk A & B.")
print(f"Checkpoint diperbarui: {ids_baru['id_kalimat'].nunique()} kalimat total terpilih (kumulatif).")

## Cara menambah data (incremental)

Bila 3000 ternyata kurang, **cukup naikkan `TARGET_TOTAL`** (mis. 3500) lalu jalankan ulang notebook ini dari atas. Notebook akan:
1. membaca `sudah_terpilih.csv` (semua id batch sebelumnya),
2. mengambil sampel baru **hanya dari kalimat yang belum pernah terpilih**,
3. menyimpan sheet `to_annotate_batch2_*.csv` dan memperbarui checkpoint.

Dengan begitu tidak ada kalimat yang terpilih dua kali, dan hasil anotasi batch lama tetap valid.

---
## **Gabung label & siapkan data prediksi (Fase 07)**

Jalankan bagian ini **setelah** sheet batch selesai dianotasi. Alur:
1. Kamu **duplikat** `to_annotate_batchN_*.csv` -> isi kolom `sentimen` (dan `aspek_salah` bila perlu) -> simpan sebagai `labeled_batchN_*.csv` di folder `data/annotation/`.
2. Sel di bawah membaca **semua** `labeled_batch*.csv`, menggabung + buang duplikat `id_kalimat` -> `labeled_all.csv`.
3. Sisa kalimat ber-aspek yang **belum** berlabel -> `untuk_prediksi.csv` (dipakai model terbaik di Fase 07).

File `labeled_batchN_*.csv` milikmu **tidak pernah ditimpa** notebook; label lama aman walau menambah batch.

In [ ]:
import glob

# baca semua file hasil anotasi milikmu (pola: labeled_batch*.csv)
label_files = sorted(glob.glob(str(ANNDIR / "labeled_batch*.csv")))
print("File anotasi ditemukan:", [Path(f).name for f in label_files] or "(belum ada)")

if label_files:
    parts = []
    for f in label_files:
        d = pd.read_csv(f, dtype={"id_kalimat": str, "id_artikel": str})
        need = {"id_kalimat", "sentimen"}
        if not need.issubset(d.columns):
            raise ValueError(f"{Path(f).name} wajib punya kolom {need}")
        parts.append(d)
    labeled = pd.concat(parts, ignore_index=True)

    # buang baris yang ditandai aspek salah (bila kolomnya ada & terisi)
    if "aspek_salah" in labeled.columns:
        n_salah = labeled["aspek_salah"].astype(str).str.strip().str.lower().isin(["x", "ya", "1", "true"]).sum()
        labeled = labeled[~labeled["aspek_salah"].astype(str).str.strip().str.lower().isin(["x", "ya", "1", "true"])]
        print(f"Baris ditandai aspek_salah dibuang: {n_salah}")

    # hanya sentimen valid
    labeled["sentimen"] = labeled["sentimen"].astype(str).str.strip().str.lower()
    valid = {"positif", "netral", "negatif"}
    tak_valid = (~labeled["sentimen"].isin(valid)).sum()
    if tak_valid:
        print(f"Peringatan: {tak_valid} baris sentimen kosong/tak valid (dilewati).")
    labeled = labeled[labeled["sentimen"].isin(valid)]

    # DEDUP: satu id_kalimat satu label (bila ada di 2 batch, ambil yang pertama)
    sebelum = len(labeled)
    labeled = labeled.drop_duplicates(subset="id_kalimat", keep="first").reset_index(drop=True)
    print(f"Label tergabung: {sebelum} -> {len(labeled)} setelah dedup id_kalimat")
else:
    labeled = pd.DataFrame(columns=["id_kalimat", "sentimen"])
    print("Belum ada file anotasi; lewati penggabungan.")

In [ ]:
if len(labeled):
    # lengkapi kolom konteks dari sumber (jaga-jaga file anotasi tak lengkap)
    meta = df.set_index("id_kalimat")
    gab = labeled.set_index("id_kalimat")
    for col in ["id_artikel", "bank", "timestamp", "kalimat", "nama_aspek", "kategori_fiqa"]:
        if col not in gab.columns or gab[col].isna().any():
            gab[col] = meta[col]
    gab = gab.reset_index()

    kolom = ["id_kalimat", "id_artikel", "bank", "timestamp", "kalimat", "nama_aspek", "kategori_fiqa", "sentimen"]
    gab[kolom].to_csv(ANNDIR / "labeled_all.csv", index=False, encoding="utf-8-sig")
    print(f"[OUTPUT] labeled_all.csv -> {len(gab):,} pasangan berlabel")
    print(gab["sentimen"].value_counts().to_string())

    # DATA PREDIKSI Fase 07: kalimat ber-aspek (tanpa Lainnya) yang BELUM berlabel
    berlabel_ids = set(gab["id_kalimat"])
    pred = kandidat[~kandidat["id_kalimat"].isin(berlabel_ids)].copy()
    assert not (set(pred["id_kalimat"]) & berlabel_ids), "data prediksi bocor dgn data berlabel!"
    pred[["id_artikel", "id_kalimat", "bank", "timestamp", "kalimat", "nama_aspek", "kategori_fiqa"]].to_csv(
        ANNDIR / "untuk_prediksi.csv", index=False, encoding="utf-8-sig")
    print(f"[OUTPUT] untuk_prediksi.csv -> {len(pred):,} kalimat (tanpa tumpang tindih dgn berlabel)")
else:
    print("Lewati: belum ada label untuk digabung.")